# 04. Обучение Conditional VAE на эмбеддингах

Бейзлайн для сравнения с WGAN-GP. Та же задача, другая семья моделей.

Требование курса: минимум 2 генеративные модели.


In [ ]:
# === Colab setup (skip if running locally) ===
import os, sys

IN_COLAB = "COLAB_GPU" in os.environ
if IN_COLAB:
    # Клонируем репо (первый раз) и переходим в него
    if not os.path.exists("/content/gan-reviews"):
        !git clone https://github.com/USER/gan-reviews.git /content/gan-reviews
    %cd /content/gan-reviews
    !pip install -q -r requirements-trainer.txt

# Добавляем корень репо в sys.path, чтобы импорты работали из notebooks/
sys.path.insert(0, os.path.abspath(".."))

# Mount Google Drive для сохранения чекпоинтов (опционально)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT_DIR = "/content/drive/MyDrive/gan-reviews/checkpoints"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
else:
    CHECKPOINT_DIR = "../checkpoints"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("CHECKPOINT_DIR =", CHECKPOINT_DIR)


In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from pathlib import Path

from models.vae import CVAE, train_vae, sample_vae

device = 'cuda' if torch.cuda.is_available() else 'cpu'
EMB_DIR = Path('../data/embeddings')
X = np.load(EMB_DIR / 'X_cls.npy').astype('float32')
y = np.load(EMB_DIR / 'labels.npy')
mean = np.load(EMB_DIR / 'X_cls_mean.npy')
std = np.load(EMB_DIR / 'X_cls_std.npy')
X_norm = (X - mean) / std

ds = TensorDataset(torch.from_numpy(X_norm), torch.from_numpy(y))
dl = DataLoader(ds, batch_size=64, shuffle=True, drop_last=True)


In [ ]:
NUM_CLASSES = int(y.max() + 1)
vae = CVAE(emb_dim=768, latent_dim=64, num_classes=NUM_CLASSES)

history = train_vae(vae, dl, n_epochs=100, lr=1e-3, beta=1.0,
                    device=device, log_every=10)


In [ ]:
torch.save({
    'vae': vae.state_dict(),
    'history': history,
    'num_classes': NUM_CLASSES,
}, f'{CHECKPOINT_DIR}/vae.pth')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history['loss']); ax[0].set_title('Total loss')
ax[1].plot(history['recon'], label='recon')
ax[1].plot(history['kl'], label='kl')
ax[1].legend(); ax[1].set_title('Recon vs KL')
plt.savefig('../results/vae_losses.png', dpi=120)
plt.show()


In [ ]:
fake, fake_labels = sample_vae(vae, n_per_class=500,
                                num_classes=NUM_CLASSES, device=device)
fake_denorm = fake.cpu().numpy() * std + mean
np.save('../data/embeddings/X_gen_vae.npy', fake_denorm)
np.save('../data/embeddings/y_gen_vae.npy', fake_labels.cpu().numpy())
print('saved', fake_denorm.shape)
